In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from datasets import Dataset
from tqdm import tqdm
import json
import os
import time
import evaluate
import csv

# -----------------------
# CONFIG
# -----------------------
OUTPUT_ROOT = "Cross_Validation_2_FineTune_155k"
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 64  # adjust for memory
CSV_OUTPUT_DIR = "fold_eval_results"
os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)

# -----------------------
# Load CV summary to get all folds
# -----------------------
summary_path = os.path.join(OUTPUT_ROOT, "cv_summary.json")
with open(summary_path, "r", encoding="utf-8") as f:
    fold_summaries = json.load(f)

fold_dirs = [os.path.join(OUTPUT_ROOT, f"fold_{i+1}") for i in range(len(fold_summaries))]

# -----------------------
# Function to evaluate one dataset on a given model
# -----------------------
def evaluate_dataset_on_model(model, tokenizer, file_path, separator="++++$++++"):
    en_sentences, te_sentences = [], []
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    if separator in lines[0]:
        for line in lines:
            if separator in line:
                en, te = line.strip().split(separator)
                en_sentences.append(en)
                te_sentences.append(te)
    else:
        if len(lines) % 2 != 0:
            lines = lines[:-1]
        for i in range(0, len(lines), 2):
            en_sentences.append(lines[i].strip())
            te_sentences.append(lines[i+1].strip())

    dataset = Dataset.from_dict({"en": en_sentences, "te": te_sentences})

    translator = pipeline(
        "translation",
        model=model,
        tokenizer=tokenizer,
        device=-1  # CPU; use 0 if GPU available
    )

    preds, refs = [], []
    total_batches = len(dataset) // BATCH_SIZE + 1

    for i in tqdm(range(0, len(dataset), BATCH_SIZE), total=total_batches, desc=f"Translating {os.path.basename(file_path)}"):
        batch = dataset[i:i+BATCH_SIZE]["en"]
        outputs = translator(batch, max_length=MAX_TARGET_LENGTH)
        preds.extend([o["translation_text"] for o in outputs])
        refs.extend([[r] for r in dataset[i:i+BATCH_SIZE]["te"]])

    bleu = evaluate.load("sacrebleu")
    chrf = evaluate.load("chrf")
    bleu_score = bleu.compute(predictions=preds, references=refs)
    chrf_score = chrf.compute(predictions=preds, references=refs)

    return {"BLEU": bleu_score["score"], "chrF": chrf_score["score"]}

# -----------------------
# TEST FILES
# -----------------------
test_files = {
    "500_sentences": "500_data.txt",
    "700_sentences": "700_data.txt"
}

# -----------------------
# Evaluate all folds per dataset & save CSV
# -----------------------
all_results = {name: {} for name in test_files}

for dataset_name, file_path in test_files.items():
    print(f"\n==== Evaluating dataset: {dataset_name} ====")
    csv_file_path = os.path.join(CSV_OUTPUT_DIR, f"{dataset_name}_folds_results.csv")
    
    with open(csv_file_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Fold", "BLEU", "chrF"])

        for fold_idx, fold_dir in enumerate(fold_dirs, start=1):
            print(f"\n--- Fold {fold_idx} ---")
            tokenizer = AutoTokenizer.from_pretrained(fold_dir)
            model = AutoModelForSeq2SeqLM.from_pretrained(fold_dir)
            model.eval()
            
            metrics = evaluate_dataset_on_model(model, tokenizer, file_path)
            all_results[dataset_name][f"fold_{fold_idx}"] = metrics
            print(f"✅ Fold {fold_idx} metrics: {metrics}")
            
            writer.writerow([fold_idx, metrics["BLEU"], metrics["chrF"]])

    print(f"\n✅ Saved CSV for {dataset_name}: {csv_file_path}")

print("\n✅ Evaluation complete for all folds and datasets.")


==== Evaluating dataset: 500_sentences ====

--- Fold 1 ---


Device set to use cpu
Translating 500_data.txt: 100%|██████████████████████████████████████████████████████████| 8/8 [10:38<00:00, 79.82s/it]


✅ Fold 1 metrics: {'BLEU': 3.778679478766941, 'chrF': 28.018798844901333}

--- Fold 2 ---


Device set to use cpu
Translating 500_data.txt: 100%|██████████████████████████████████████████████████████████| 8/8 [09:52<00:00, 74.04s/it]


✅ Fold 2 metrics: {'BLEU': 3.599187835222342, 'chrF': 27.984256103214623}

--- Fold 3 ---


...

✅ Fold 3 metrics: {'BLEU': 69.4960160175163, 'chrF': 85.07090894819551}

--- Fold 4 ---


Device set to use cpu
Translating 700_data.txt: 100%|████████████████████████████████████████████████████████| 11/11 [06:33<00:00, 35.76s/it]


✅ Fold 4 metrics: {'BLEU': 68.31024636945698, 'chrF': 84.44296310553118}

--- Fold 5 ---


Device set to use cpu
Translating 700_data.txt: 100%|████████████████████████████████████████████████████████| 11/11 [06:14<00:00, 34.00s/it]


✅ Fold 5 metrics: {'BLEU': 69.70465281867324, 'chrF': 85.09868234622685}

✅ Saved CSV for 700_sentences: fold_eval_results\700_sentences_folds_results.csv

✅ Evaluation complete for all folds and datasets.
